# Cohort B: link cancer registry diagnoses to EHR diagnoses (Cartesian/crosswalk setup)

Same linkage approach as `01_set_up_cohort_a_cartesian`, applied to **Cohort B**. Builds the WHO
site/histology crosswalk, matches registry diagnoses to EHR diagnoses, and classifies each EHR
diagnosis into a `Rollup2` cancer category. Two differences from Cohort A's version:

- The "EHR only, no registry" case here is folded directly into `episode_matching` (Cohort B does
  not carry the separate "registry only" branch used for Cohort A).
- The final `cohort_b_derived.cohort_b_source` table additionally excludes patients flagged in
  `registries.crstar_missing_patients` if their case is analytic (`CLASS_OF_CASE_N610 > 22`).

Downstream notebooks: `03_cohort_b_care_site_assignment` and `06_set_up_cohort_b_feature_table`.

In [ ]:
USE CATALOG your_catalog;

### Reference crosswalk: WHO site/histology codes
Same reference view as in the Cohort A notebook (recreated here since notebooks in this pipeline
are run independently).

In [ ]:
%sql 
drop view if exists WHOCartesian;
CREATE view WHOCartesian as
SELECT sc.*, hc.SiteRange, hc.HistologyExplode
FROM reference.who_site_codes sc LEFT JOIN reference.who_histology_codes hc ON sc.Category = hc.Category

### Registry patients (Cohort A)
Pulls CIPOC registry diagnoses for patients present in `cohort_a.person` restricted to diagnoses on/after the ICD-10 transition date for use in excluding these patients later in this notebook. 

In [ ]:
%sql
drop view if exists registry_pts;
create view registry_pts as 
--grab the crosswalk, limit to post-ICD-10 time range
select p.person_id, cx.cancer_site as registry_site, cx.date_of_diagnosis, cx.primary_site as registry_dx_code, cx.sequence_number_central   
from linkage_files.cipoc_xwalk cx JOIN cohort_a.person p ON cx.pat_id = p.person_source_value
where cx.date_of_diagnosis >= '2015-10-01'

### EHR condition codes present in the data, mapped back to ICD-10-CM
Same approach as Cohort A, but joined against Cohort B's `condition_occurrence` table.

In [ ]:
%sql
--convert condition_concept_ids that are actually in the data back to ICD-10s
drop view if exists all_c_codes_in_data;
create view all_c_codes_in_data as 
SELECT distinct c.concept_id as original_concept_id, c2.*
FROM cohort_a.concept c JOIN cohort_b.condition_occurrence co ON c.concept_id = co.condition_concept_id 
    JOIN cohort_a.concept_relationship cr ON c.concept_id = cr.concept_id_2 and relationship_id = 'Maps to'
    JOIN cohort_a.concept c2 ON c2.concept_id = cr.concept_id_1 and c2.vocabulary_id = 'ICD10CM' and c2.concept_code LIKE 'C%' and c2.invalid_reason is null

### Classify each EHR cancer diagnosis into a Rollup2 category

In [ ]:
%sql
drop view if exists cancer_code_density;
create view cancer_code_density as 
--compile EHR cancers
with master_code_list as (
    select cast(ExplodedSiteRange as string), Rollup, Rollup2 from WHOCartesian
    UNION
    select replace(replace(ICDCode,'.',''),'C','') as ExplodedSiteRange, Rollup, Rollup2 from reference.who_ehr_mapping
),

prelim_cancers as (
SELECT distinct co.person_id as ehr_person_id, co.condition_occurrence_id, co.condition_concept_id,  
case when length(left(replace(a.concept_code,'.',''),4)) = 3 then left(replace(a.concept_code,'.',''),4) || '0'  else left(replace(a.concept_code,'.',''),4) end as truncated_stripped_icd_ehr, co.condition_start_date
FROM cohort_b.condition_occurrence co 
JOIN all_c_codes_in_data a ON co.condition_concept_id = a.original_concept_id
)

select p.*, 
case when m_ehr.Rollup2 is null then 'Unmapped' else m_ehr.Rollup2 end as Rollup2 
from prelim_cancers p LEFT JOIN master_code_list m_ehr ON replace(replace(truncated_stripped_icd_ehr,'.',''),'C','') = m_ehr.ExplodedSiteRange

### Collapse diagnosis codes into EHR "episodes"

In [ ]:
%sql
drop view if exists ehr_episode_dates;
create view ehr_episode_dates as 
--create cancer episodes in the EHR data
SELECT ehr_person_id, Rollup2, min(condition_start_date) as episode_start, max(condition_start_date) as episode_end
FROM cancer_code_density
group by ehr_person_id, Rollup2

### Match EHR episodes to registry diagnoses
For Cohort B, we ensure that none of the patients are in the registry by performing an anti-join with the previously defined `registry_pts` table

In [ ]:
%sql
drop view if exists episode_matching;
create view episode_matching as
--cancers in the EHR that have no registry match 
--possible reasons: MRN merges, registry cancer is pre-2015 but still mentioned in EHR
SELECT distinct co.ehr_person_id, n.person_id as registry_person_id, co.condition_occurrence_id as ehr_cancer_id, co.condition_concept_id as original_concept_id_ehr, truncated_stripped_icd_ehr, e.episode_start as ehr_episode_start, e.episode_end as ehr_episode_end, 'EHR only no registry' as match_type, e.Rollup2 as ehr_rollup2, n.date_of_diagnosis as registry_cancer_dx_date, datediff(n.date_of_diagnosis, e.episode_start) as daysbt, n.person_id || n.sequence_number_central as rownum, n.registry_site
FROM cancer_code_density co
JOIN ehr_episode_dates e ON co.ehr_person_id = e.ehr_person_id and co.Rollup2 = e.Rollup2
LEFT JOIN registry_pts n ON co.ehr_person_id = n.person_id 
WHERE n.person_id is null
and e.episode_start >= '2015-10-01'

### Persist the final Cohort B source table
Same as Cohort A's source table, with an extra filter removing patients in the CRSTAR
missing-patients list who have an analytic case (`CLASS_OF_CASE_N610 > 22`).

In [ ]:
%sql
--simplifying the output from the view above
drop table if exists cohort_b_derived.cohort_b_source;
create table cohort_b_derived.cohort_b_source as
SELECT distinct e.ehr_person_id, e.registry_person_id, e.ehr_episode_start, e.ehr_episode_end, e.match_type, e.ehr_rollup2, e.registry_cancer_dx_date, e.daysbt, e.rownum as registry_rownum, e.registry_site
FROM episode_matching e LEFT JOIN registries.crstar_missing_patients c ON e.ehr_person_id = c.PERSON_ID 
--remove patients who are in the crstar missing patients with an analytic case
WHERE c.PERSON_ID is null OR c.CLASS_OF_CASE_N610 > 22